In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
    coalesce,
    concat,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_contact_details_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ContactDetailId	String"
        #     ",SourceSystem	String"
        #     ",PortfolioKey	String"
        #     ",PortfolioName	String"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId	String"
        #     ",ContactDetailKey	String"
        #     ",CDI	string"
        #     ",PartyKey	String"
        #     ",ContactDetailTypeCode	string"
        #     ",ContactDetailType	string"
        #     ",ContactDetailTypeGroup string"
        #     ",FullAddress	string"
        #     ",Country	string"
        #     ",County	string"
        #     ",City	string"
        #     ",PostalCode	string"
        #     ",Street	string"
        #     ",Street1 string"
        #     ",Street2 string"
        #     ",Street3 string"
        #     ",StreetNumber	string"
        #     ",PhoneNumber1	string"
        #     ",PhoneNumber2	string"
        #     ",PhoneNumber3	string"
        #     ",EmailAddress1	string"
        #     ",EmailAddress2	string"
        #     ",IsValidFlag	boolean"
        #     ",CommunicationFlag	string"
        #     ",IsPrimaryFlag	boolean"
        #     ",CreatedBy	string"
        #     ",CreatedDate	timestamp"
        #     ",LastUpdatedDate	date"
        #     ",OriginalTelephoneCode	string"
        #     ",OriginalTelephoneType	string"
        #     ",IsActiveFlag	string"
        #     ",DoNotCallFlag	boolean"
        #     ",IsDialerActivityFlag	int"
        #     ",DialerActivityTelephoneNumber	string"
        #     ",InternalConfirmationTypeCode	int"
        #     ",InternalConfirmationType	string"
        #     ",CollectionSourceTypeCode	int"
        #     ",CollectionSourceType	string"
        #     ",DoNotSendFlag boolean"
        #     ",AddContactFlag boolean"
        #     ",SerialNumber int"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ContactDetails"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qc",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "vendor_code_qualco",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "cust_code",
            "cdi",
            "businessdate",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ================== customer_phone ==================

        customer_phone_extra_filters = None

        if customer_phone_extra_filters is not None:
            customer_phone_extra_filters.append(date_filter)
            customer_phone_final_filters = " AND ".join(customer_phone_extra_filters)
        else:
            customer_phone_final_filters = date_filter

        customer_phone_selected_fields = [
            "cuph_code",
            "vend_code",
            "cust_code",
            "cuph_type",
            "phone",
            "cuph_confirmation",
            "cuph_is_primary",
            "cuph_created_by",
            "cuph_created",
            "cuph_last_upd",
            "cuph_origin_type",
            "cuph_active",
            "cuph_do_not_call",
            "cuph_is_dialer",
            "cuph_dialer_phone",
            "cuph_internal_confirmation",
            "cuph_collection_source_code",
            "businessdate",
        ]

        customer_phone = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer_phone",
            customer_phone_final_filters,
            customer_phone_selected_fields,
        )

        # ================== lst_of_val ==================

        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = [
            "lov_code",
            "lov_desc",
            "businessdate",
        ]

        lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )

        # Join Conditions

        lst_of_val_internal = lst_of_val.select(
            lst_of_val.lov_code.alias("lov_code_internal"),
            lst_of_val.lov_desc.alias("lov_desc_internal"),
        )

        lst_of_val_collection = lst_of_val.select(
            lst_of_val.lov_code.alias("lov_code_collection"),
            lst_of_val.lov_desc.alias("lov_desc_collection"),
        )

        customer_phone_lst_of_val_internal_condition = (
            customer_phone["cuph_internal_confirmation"]
            == lst_of_val_internal["lov_code_internal"]
        )

        customer_phone_lst_of_val_collection_condition = (
            customer_phone["cuph_collection_source_code"]
            == lst_of_val_collection["lov_code_collection"]
        )

        customer_phone_portfolio_condition = (
            customer_phone["vend_code"] == portfolio["vendor_code_qualco"]
        )

        customer_phone_customer_condition = (
            customer_phone["cust_code"] == customer["cust_code"]
        )

        # Joins
        target_table_qc_phones = (
            customer.join(
                customer_phone,
                customer_phone_customer_condition,
                "left",
            )
            .join(
                lst_of_val_internal,
                customer_phone_lst_of_val_internal_condition,
                "left",
            )
            .join(
                lst_of_val_collection,
                customer_phone_lst_of_val_collection_condition,
                "left",
            )
            .join(
                portfolio,
                customer_phone_portfolio_condition,
                "left",
            )
            
        )

        # HashKeys
        target_table_qc_ContactDetailID_primary_key_fields = [
            customer_phone.cuph_code,
            customer_phone.cust_code,
            customer_phone.cuph_type,
            lit("qc"),
        ]

        target_table_qc_phones = target_table_qc_phones.withColumn(
            "ContactDetailId",
            sha2(
                concat_ws(";", *target_table_qc_ContactDetailID_primary_key_fields),
                256,
            ),
        ).withColumn(
            "ContactDetailKey",
            concat_ws(
                ";",
                customer_phone.cuph_code,
                customer_phone.cust_code,
                customer_phone.cuph_type,
            ),
        )

        # ================== Selection of Final Columns ==================

        contact_details_qc_df_phones = target_table_qc_phones.select(
            target_table_qc_phones.ContactDetailId.cast("String").alias(
                "ContactDetailId"
            ),
            lit("qc").cast("String").alias("SourceSystem"),
            customer_phone.vend_code.cast("String").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table_qc_phones.ContactDetailKey.cast("String"),
            customer.cdi.cast("String").alias("CDI"),
            customer_phone.cust_code.cast("String").alias("PartyKey"),
            customer_phone.cuph_type.cast("String").alias("ContactDetailTypeCode"),
            lit(None).cast("String").alias("ContactDetailType"),
            lit("Phone").cast("String").alias("ContactDetailTypeGroup"),
            lit(None).cast("String").alias("FullAddress"),
            lit(None).cast("String").alias("Country"),
            lit(None).cast("String").alias("County"),
            lit(None).cast("String").alias("City"),
            lit(None).cast("String").alias("PostalCode"),
            lit(None).cast("String").alias("Street"),
            lit(None).cast("String").alias("Street1"),
            lit(None).cast("String").alias("Street2"),
            lit(None).cast("String").alias("Street3"),
            lit(None).cast("String").alias("StreetNumber"),
            customer_phone.phone.cast("String").alias("PhoneNumber1"),
            lit(None).cast("String").alias("PhoneNumber2"),
            lit(None).cast("String").alias("PhoneNumber3"),
            lit(None).cast("String").alias("EmailAddress1"),
            lit(None).cast("String").alias("EmailAddress2"),
            customer_phone.cuph_confirmation.cast("boolean").alias("IsValidFlag"),
            lit(None).cast("String").alias("CommunicationFlag"),
            customer_phone.cuph_is_primary.cast("boolean").alias("IsPrimaryFlag"),
            customer_phone.cuph_created_by.cast("String").alias("CreatedBy"),
            customer_phone.cuph_created.cast("timestamp").alias("CreatedDate"),
            customer_phone.cuph_last_upd.cast("date").alias("LastUpdatedDate"),
            customer_phone.cuph_origin_type.cast("String").alias(
                "OriginalTelephoneCode"
            ),
            customer_phone.cuph_origin_type.cast("String").alias(
                "OriginalTelephoneType"
            ),
            customer_phone.cuph_active.cast("String").alias("IsActiveFlag"),
            customer_phone.cuph_do_not_call.cast("boolean").alias("DoNotCallFlag"),
            customer_phone.cuph_is_dialer.cast("int").alias("IsDialerActivityFlag"),
            customer_phone.cuph_dialer_phone.cast("String").alias(
                "DialerActivityTelephoneNumber"
            ),
            customer_phone.cuph_internal_confirmation.cast("int").alias(
                "InternalConfirmationTypeCode"
            ),
            lst_of_val_internal.lov_desc_internal.cast("String").alias(
                "InternalConfirmationType"
            ),
            customer_phone.cuph_collection_source_code.cast("int").alias(
                "CollectionSourceTypeCode"
            ),
            lst_of_val_collection.lov_desc_collection.cast("String").alias(
                "CollectionSourceType"
            ),
            lit(None).alias("DoNotSendFlag"),
            lit(None).alias("AddContactFlag"),
            lit(None).alias("SerialNumber"),
            customer_phone.businessdate.alias("BusinessDate"),
        )

        # ================== customer_address ==================

        customer_address_extra_filters = None

        if customer_address_extra_filters is not None:
            customer_address_extra_filters.append(date_filter)
            customer_address_final_filters = " AND ".join(
                customer_address_extra_filters
            )
        else:
            customer_address_final_filters = date_filter

        customer_address_selected_fields = [
            "cuad_code",
            "vend_code",
            "cust_code",
            "cuad_confirmation",
            "cuad_created",
            "cuad_last_upd",
            "cuad_active",
            "cuad_internal_confirmation",
            "cuad_collection_source_code",
            "cuad_address_type",
            "cuad_city",
            "cuad_postcode",
            "cuad_country",
            "cuad_street_1",
            "cuad_street_2",
            "cuad_street_3",
            "cuad_street_num",
            "cuad_do_not_send_flag",
            "cuad_add_contact_flag",
            "BusinessDate",
        ]

        customer_address = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer_address",
            customer_address_final_filters,
            customer_address_selected_fields,
        )

        # Join Conditions

        customer_address_lst_of_val_internal_condition = (
            customer_address["cuad_internal_confirmation"]
            == lst_of_val_internal["lov_code_internal"]
        )

        customer_address_lst_of_val_collection_condition = (
            customer_address["cuad_collection_source_code"]
            == lst_of_val_collection["lov_code_collection"]
        )

        customer_address_portfolio_condition = (
            customer_address["vend_code"] == portfolio["vendor_code_qualco"]
        )

        customer_address_customer_condition = (
            customer_address["cust_code"] == customer["cust_code"]
        )

        # Joins
        target_table_qc_address = (
            customer.join(
                customer_address,
                customer_address_customer_condition,
                "left",
            )
            .join(
                lst_of_val_internal,
                customer_address_lst_of_val_internal_condition,
                "left",
            )
            .join(
                lst_of_val_collection,
                customer_address_lst_of_val_collection_condition,
                "left",
            )
            .join(
                portfolio,
                customer_address_portfolio_condition,
                "left",
            )
           
        )

        # HashKeys
        target_table_qc_ContactDetailID_primary_key_fields = [
            customer_address.cuad_code,
            customer_address.cust_code,
            customer_address.cuad_address_type,
            lit("qc"),
        ]

        contact_details_full_address_fields = [
            customer_address.cuad_street_1,
            customer_address.cuad_street_num,
            customer_address.cuad_city,
            customer_address.cuad_postcode,
        ]

        target_table_qc_address = (
            target_table_qc_address.withColumn(
                "ContactDetailId",
                sha2(
                    concat_ws(";", *target_table_qc_ContactDetailID_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ContactDetailKey",
                concat_ws(
                    ";",
                    customer_address.cuad_code,
                    customer_address.cust_code,
                    customer_address.cuad_address_type,
                ),
            )
            .withColumn(
                "fulladdress",
                concat_ws(";", *contact_details_full_address_fields),
            )
            .withColumn(
                "street",
                when(
                    ~coalesce(col("cuad_street_3"), lit("N/A")).isin("N/A", "", "-") &
                    ~coalesce(col("cuad_street_3"), lit("N/A")).rlike("(?i)street"),
                    col("cuad_street_3")
                )
                .when(
                    coalesce(col("cuad_street_1"), lit("N/A")).isin("N/A", "", "-"),
                    col("cuad_street_2")
                )
                .when(
                    (col("cuad_street_1").cast("string") != col("cuad_street_2").cast("string")) &
                    ~coalesce(col("cuad_street_2"), lit("N/A")).isin("N/A", "", "-") &
                    ~coalesce(col("cuad_street_2"), lit("N/A")).rlike("(?i)street"),
                    concat(
                        col("cuad_street_1").cast("string"), 
                        lit(" "), 
                        col("cuad_street_2").cast("string")
                    )
                ).otherwise(col("cuad_street_1"))
            )
        ) 

        # ================== Selection of Final Columns ==================

        contact_details_qc_df_address = target_table_qc_address.select(
            target_table_qc_address.ContactDetailId.cast("String").alias(
                "ContactDetailId"
            ),
            lit("qc").alias("SourceSystem"),
            customer_address.vend_code.cast("String").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            target_table_qc_address.ContactDetailKey.cast("string"),
            customer.cdi.cast("String").alias("CDI"),
            customer_address.cust_code.cast("string").alias("PartyKey"),
            target_table_qc_address.cuad_address_type.cast("string").alias(
                "ContactDetailTypeCode"
            ),
            target_table_qc_address.cuad_address_type.cast("string").alias(
                "ContactDetailType"
            ),
            lit("Address").cast("String").alias("ContactDetailTypeGroup"),
            target_table_qc_address.fulladdress.alias("FullAddress"),
            target_table_qc_address.cuad_country.alias("Country"),
            lit(None).alias("County"),
            target_table_qc_address.cuad_city.alias("City"),
            target_table_qc_address.cuad_postcode.alias("PostalCode"),
            target_table_qc_address.street.alias("Street"),
            target_table_qc_address.cuad_street_1.alias("Street1"),
            target_table_qc_address.cuad_street_2.alias("Street2"),
            target_table_qc_address.cuad_street_3.alias("Street3"),
            target_table_qc_address.cuad_street_num.alias("StreetNumber"),
            lit(None).alias("PhoneNumber1"),
            lit(None).alias("PhoneNumber2"),
            lit(None).alias("PhoneNumber3"),
            lit(None).alias("EmailAddress1"),
            lit(None).alias("EmailAddress2"),
            target_table_qc_address.cuad_confirmation.alias("IsValidFlag"),
            target_table_qc_address.cuad_internal_confirmation.cast("String").alias(
                "CommunicationFlag"
            ),
            lit(None).alias("IsPrimaryFlag"),
            lit(None).alias("CreatedBy"),
            target_table_qc_address.cuad_created.alias("CreatedDate"),
            target_table_qc_address.cuad_last_upd.cast("date").alias("LastUpdatedDate"),
            lit(None).alias("OriginalTelephoneCode"),
            lit(None).alias("OriginalTelephoneType"),
            target_table_qc_address.cuad_active.cast("string").alias("IsActiveFlag"),
            lit(None).alias("DoNotCallFlag"),
            lit(None).alias("IsDialerActivityFlag"),
            lit(None).alias("DialerActivityTelephoneNumber"),
            target_table_qc_address.cuad_internal_confirmation.alias(
                "InternalConfirmationTypeCode"
            ),
            lst_of_val_internal.lov_desc_internal.cast("String").alias(
                "InternalConfirmationType"
            ),
            target_table_qc_address.cuad_collection_source_code.cast("int").alias(
                "CollectionSourceTypeCode"
            ),
            lst_of_val_collection.lov_desc_collection.cast("String").alias(
                "CollectionSourceType"
            ),
            target_table_qc_address.cuad_do_not_send_flag.alias("DoNotSendFlag"),
            target_table_qc_address.cuad_add_contact_flag.alias("AddContactFlag"),
            lit(None).alias("SerialNumber"),
            customer_address.BusinessDate.alias("BusinessDate"),
        )

        # ================== Union ==================

        # Union the DataFrames
        contact_details_qc = contact_details_qc_df_address.union(
            contact_details_qc_df_phones
        )

        contact_details_qc.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_contact_details_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)